Run the setup cell first, then restart the kernel if prompted and run the remaining cells in order. This copy has corrected source and cleared outputs; it has not been executed against the downloaded dataset. The other required packages are numpy, pandas, matplotlib, and scikit-learn. GitHub displays saved notebook contents and does not execute these cells or install packages. Run and save the notebook locally or in Colab before uploading if you want results displayed.

In [ ]:
# Install certifi into the active notebook kernel before importing it.
%pip install certifi

In [ ]:
# SC5002 - LAB ASSIGNMENT PART 2: LOW-CODE ML TEMPLATE
# Run the setup cell above, then run these cells in order.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import ssl
import certifi

# Use the CA bundle to verify HTTPS when downloading the dataset.
ssl._create_default_https_context = lambda: ssl.create_default_context(
    cafile=certifi.where()
)

In [ ]:
# --- TASK 2.1: DATASET SETUP ---
print("=== TASK 2.1: Loading California Housing Dataset ===")
housing = fetch_california_housing(as_frame=True)
df = housing.frame.sample(n=1000, random_state=42)

print("First 5 rows of dataset (Copy these for AI Prompt 2.1):")
print(df.head())

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Scaling happens inside each cross-validation fold in Task 2.2.

In [ ]:
# --- TASK 2.2: MODEL TRAINING & CROSS-VALIDATION ---
print("=== TASK 2.2: Training Linear Regression ===")
lin_reg = make_pipeline(StandardScaler(), LinearRegression())
lin_scores = cross_val_score(lin_reg, X_train, y_train, cv=5, scoring='r2')
print(f"Linear Regression Mean 5-Fold R2 Score: {lin_scores.mean():.4f}")

# Ridge Regression Experiments
alphas = [0.1, 10.0, 1000.0]
ridge_results = {}
print("--- Training Ridge Regression with Varying Alpha Values ---")
for a in alphas:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=a))
    scores = cross_val_score(ridge, X_train, y_train, cv=5, scoring='r2')
    ridge_results[a] = scores.mean()
    print(f"Ridge Regression (alpha={a:6.1f}) Mean 5-Fold R2 Score: {scores.mean():.4f}")

In [ ]:
# --- TASK 2.3: VISUALIZATION ---
print("=== TASK 2.3: Generating Model Comparison Chart ===")
plt.figure(figsize=(9, 5))
models = ['Linear Reg'] + [f'Ridge (a={a})' for a in alphas]
scores = [lin_scores.mean()] + list(ridge_results.values())
bars = plt.bar(models, scores, color=['#3182ce', '#38a169', '#dd6b20', '#e53e3e'])
plt.ylabel('Mean 5-Fold Cross-Validation R² Score', fontsize=10)
plt.title('Linear vs. Ridge Regression Performance Comparison', fontsize=12, fontweight='bold')
plt.ylim(min(0, min(scores) - 0.1), max(1.0, max(scores) + 0.1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02,
             f'{yval:.3f}', ha='center', va='bottom')
plt.show()